In [ ]:
! pip install spacy ebooklib pandas nltk bs4


In [ ]:
! python -m spacy download pt_core_news_sm

In [1]:
import ebooklib
import re
from ebooklib import epub as ep
import pandas as pd
from html.parser import HTMLParser
from bs4 import BeautifulSoup
import csv
import spacy
from nltk.corpus import stopwords
from collections import Counter, defaultdict


In [2]:
book = ep.read_epub('resources/historia_europa.epub')


/home/vinicius/workspace/book_kg/.venv/lib/python3.10/site-packages/ebooklib/epub.py:1395: UserWarning: In the future version we will turn default option ignore_ncx to True.
  warnings.warn('In the future version we will turn default option ignore_ncx to True.')
/home/vinicius/workspace/book_kg/.venv/lib/python3.10/site-packages/ebooklib/epub.py:1423: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/xmlns:rootfile[@media-type]'
  for root_file in tree.findall('//xmlns:rootfile[@media-type]', namespaces={'xmlns': NAMESPACES['CONTAINERNS']}):


In [8]:
def chap2text(chap):
    output = ''
    soup = BeautifulSoup(chap, 'html.parser')
    text = soup.find_all(text=True)
    for t in text:
        if t.parent.name not in blacklist:
            output += '{} '.format(t)
    return output.strip()

blacklist = ['[document]', 'noscript', 'header', 'html', 'meta', 'head', 'input', 'script']

chapters = []
for item in book.get_items():
    if item.get_type() == ebooklib.ITEM_DOCUMENT:
        chapters.append(item.get_content())

with open('book_chapters.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Chapter Index', 'Chapter Number', 'Chapter Content'])  # Write header
    
    for i, chapter in enumerate(chapters, 1):
        text = chap2text(chapter)
        text_lines = text.split('\n')
        
        if len(text_lines) >= 3:  # Ensure we have at least 3 lines
            chapter_name = ' '.join(text_lines[:1]).strip()  # First two lines for chapter name
            chapter_content = '\n'.join(text_lines[1:]).strip()  # Rest for content
            
            # Only write non-empty chapters
            if chapter_name and chapter_content:
                writer.writerow([i, chapter_name, chapter_content])

/tmp/ipykernel_325798/1855634191.py:4: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  text = soup.find_all(text=True)


In [4]:
nlp = spacy.load('pt_core_news_sm')
stop_words = set(stopwords.words('portuguese'))
currency_pattern = re.compile(r"^[1-9][0-9,]*(\.[0-9]{1,2})?$")
letter_point_pattern = re.compile(r'^(?!^[a-zA-Z]+\.$).*$')

In [5]:
def preprocess_token(token):
    return token.lemma_.strip().lower()

def is_token_allowed(token):
    return bool(
        token
        and str(token).strip()
        and str(token).strip().lower() != "r$"
        and str(token).strip().lower() != "us$"
        and str(token).strip().lower() != "\n"
        and str(token).strip().lower() != "\n "
        and str(token).strip().lower() != "$"
        and str(token).strip().lower() != "+"
        and not currency_pattern.match(str(token).strip())
        and letter_point_pattern.match(str(token).strip())
        and not token.is_stop
        and not token.is_punct
        and token not in stop_words
    )

def filter_tokens(doc)-> dict:
    return [preprocess_token(token) for token in doc if is_token_allowed(token)]

def ner(d):
    doc = nlp(d)
    ners = []
    for token in doc:
        if token.ent_type_ != "" and is_token_allowed(token):
            ners.append({"token": str(token), "processed_token": preprocess_token(token), "entity_type":token.ent_type_})
    return ners_frequency(ners)


def ners_frequency(ners, entity_type = None):
    # print(ners)
    entity_counts = Counter()
    token_counts = defaultdict(Counter)
    for n in ners:
        entity_type = n['entity_type']
        token = n['token']

        # Increment the count for the entity_type
        entity_counts[entity_type] += 1

        # Increment the count for the token within its entity_type
        token_counts[entity_type][token] += 1

    # Convert defaultdict to regular dict for better printing
    token_counts = dict(token_counts)

    return {
        'entity_counts': dict(entity_counts),
        'token_counts': token_counts
    }

def flatten_ner_results(ner_result):
    flattened = {}
    if isinstance(ner_result, dict):
        for entity_type, count in ner_result['entity_counts'].items():
            flattened[f'Entity_{entity_type}'] = count
        for entity_type, tokens in ner_result['token_counts'].items():
            for token, count in tokens.items():
                flattened[f'{entity_type}_{token}'] = count
    return flattened

In [9]:
d = pd.read_csv('book_chapters.csv')


In [10]:
d['NER_Results'] = d['Chapter Content'].apply(ner)

In [11]:
# Apply flattening to each row
d['Flattened_NER'] = d['NER_Results'].apply(flatten_ner_results)

# Create a summary DataFrame
summary_df = pd.DataFrame(d['Flattened_NER'].tolist()).fillna(0)

# Add the summary to the original DataFrame
d = pd.concat([d, summary_df], axis=1)

# Calculate overall summary
overall_summary = summary_df.sum().sort_values(ascending=False)

# Print results
print("Summary per chapter:")
print(d[['Chapter Index', 'Chapter Number'] + list(summary_df.columns)])

print("\nOverall summary:")
print(overall_summary)

Summary per chapter:
    Chapter Index                                    Chapter Number  \
0               3  Título original:  The Shortest History of Europe   
1               4                                        INTRODUÇÃO   
2               5   CAPÍTULO 1.   Europa clássica e Europa medieval   
3               6                      CAPÍTULO 2.   Europa moderna   
4               7                 INTERLÚDIO.   O toque do clássico   
5               8               CAPÍTULO 3.   Invasões e conquistas   
6               9                 CAPÍTULO 4.   Formas de governo I   
7              10                CAPÍTULO 5.   Formas de governo II   
8              11                 CAPÍTULO 6.   Imperadores e papas   
9              12                             CAPÍTULO 7.   Línguas   
10             13                   CAPÍTULO 8.   As pessoas comuns   
11             14                   INTERLÚDIO.   Por que a Europa?   
12             15                                FORÇAS 

In [12]:
filtered_columns = [col for col in summary_df.columns if col.startswith('PER_')]
t_sum = summary_df[filtered_columns].transpose()
t_sum.columns = d['Chapter Number']
t_sum = t_sum.iloc[:, 5:]
t_sum = t_sum.reset_index()